In [1]:
import xarray as xr, numpy as np, pandas as pd
import matplotlib.pyplot as plt, cartopy.crs as ccrs
import matplotlib.cm as cm, matplotlib.colors as mcolors
from matplotlib.patches import Patch
import cartopy.feature as cfeature
import pickle 

from functions import verify_population_coverage, EU_MEMBERS, \
                        country_weights_dict, baseline_start, baseline_end, lat_to_str_dict, \
                        cider_inj_lats_popweighted, basepath, cpath, cider_combined, pop, \
                        select_eu_mode

In [2]:
n_members = 500
newpath = basepath + f'{n_members}_members/' 
spath = newpath + 'simple_memory_model/'

ssp_plus_invert = xr.open_dataset(newpath + 
                    f'SSP370_plus_InVERT_{n_members}member_ensemble_TREFHT_2015_to_2065.nc')

W_norm_eu = country_weights_dict['W_norm_eu']
countries_idx_eu = country_weights_dict['countries_idx_eu']

In [3]:
baseline = ssp_plus_invert.sel(time=slice(baseline_start, baseline_end))

countries = country_weights_dict['countries_gdf']
verify_population_coverage(country_weights_dict)

PASSED -- Population coverage: 100.000000% (8.016736e+09 of 8.016736e+09 people-equivalent) accounted for across kept country rows.


{'population_total': 8016736089.992069,
 'population_accounted': 8016736089.992069,
 'coverage_frac': 1.0,
 'passed': True}

In [4]:
#### Values of sensitivity-tested parameters (DEFAULT)
#### Only using these to find path where an existing min_injection_rates dataset is already saved
#### to use the countries etc. 

buffer_months = 12; tau = 12; eval_months = 36

### Where results are saved for this set of parameters
rpath = newpath + \
        f'results_buffer={buffer_months}m_tau={tau}m_eval={eval_months}m/'
print(rpath)

k_setting = 'country_k'
## Load min injection rate datasets computed in 05_find_min_injection_rate_..._.py
inj_rates_single_lat = xr.open_dataset(rpath + f'single_lat_min_rates_{k_setting}.nc')
inj_rates_uniform = xr.open_dataset(rpath + f'global_uniform_min_rates_{k_setting}.nc')

## Filter EU countries according to the chosen eu_mode
eu_mode = 'aggregated'
inj_rates_single_lat = select_eu_mode(inj_rates_single_lat, eu_mode)
inj_rates_uniform = select_eu_mode(inj_rates_uniform, eu_mode)

/home/msaenger/Perceptions_2025/cleaned_code/500_members/results_buffer=12m_tau=12m_eval=36m/


In [5]:
## --- Example configuration ---

deployment_start_year = 2035   # Calendar year every country/ensemble member starts SAI in

buffer_months = 12             # gap, in months, between when SAI deployment begins and when
                                # a given test year begins -- i.e. how long after SAI starts
                                # before we start scoring people's perceptions of the
                                # temperatures they feel.

                                # This is NOT a zero-weighted gap. 
                                # Perceptions of "recent" temperature still include whatever
                                # they felt during the buffer. So the buffer months are folded
                                # directly into the recency-weighted reference period below,
                                # as its most-recent (least-lagged) months: if buffer_months=12,
                                # the entire buffer is the lag=1 reference year; if
                                # buffer_months=6, the buffer's 6 months plus the 6 months
                                # right before deployment together make up the lag=1
                                # reference year; if buffer_months=18, the buffer's last 12
                                # months are lag=1, and its first 6 months combine with the 6
                                # months right before deployment to make up lag=2; and so on.

n_test_years = 2               # how many consecutive years, starting immediately after the
                                # first buffer, make up the test period -- e.g. n_test_years=3
                                # asks "is each of the 3 years right after the buffer, ON
                                # AVERAGE, cooler than that year's own (moving) reference
                                # period?"

n_ref_years = 7                # how many 12-month blocks make up each test year's
                                # reference period, counting backward from that test year's
                                # own start (i.e. the reference period is always flush
                                # against the test year; the buffer's months are absorbed into
                                # whichever reference block they fall in).

                                # MOVING BASELINE: the reference period moves forward in
                                # time along with the test year it belongs to. The first
                                # test year's reference period is the n_ref_years x
                                # 12-month blocks immediately preceding it (buffer
                                # included). The second test year's reference
                                # period has moved 1 year forward -- its lag=1 block is
                                # now the first test year itself -- and so on.

## Recency weights for the n_ref_years reference years, taken from Moore et al. 2019:
##   Lag (12-month blocks before the test year starts):  1     2     3     4     5     6     7
##   Weight:                                              0.23  0.20  0.17  0.14  0.11  0.09  0.06
## find_min_injection_rate() wants these in chronological order (oldest -> most recent),
## i.e. ref_weights[0] = the block farthest before the test year (lag 7) ... ref_weights[-1] =
## the block immediately adjoining the test year (lag 1, the highest weight). 
## The same weights are reused, unchanged, for every test year's own (moved) reference period.
lag_weights = {1: 0.23, 2: 0.20, 3: 0.17, 4: 0.14, 5: 0.11, 6: 0.09, 7: 0.06}
ref_weights = [lag_weights[lag] for lag in range(n_ref_years, 0, -1)]

ref_weights

[0.06, 0.09, 0.11, 0.14, 0.17, 0.2, 0.23]

In [6]:
## --- Helper functions ---

## Find the single, shared deployment start index -- January of
## deployment_start_year -- in ssp_plus_invert's monthly time axis. This index is the
## same for every country and every ensemble member. The first test year's buffer also starts here.

_time_years = np.array([t.year for t in ssp_plus_invert.time.values])
_time_months = np.array([t.month for t in ssp_plus_invert.time.values])
start_idx = int(np.argmax((_time_years == deployment_start_year) & (_time_months == 1)))

assert _time_years[start_idx] == deployment_start_year and _time_months[start_idx] == 1, \
    f"Could not find January {deployment_start_year} in ssp_plus_invert.time"

print(f"Deployment start index: {start_idx} -> {ssp_plus_invert.time.values[start_idx]}")

## CIDER's output only covers 5 years (60 months) after deployment
CIDER_HORIZON_MONTHS = 60


def test_year_bounds(year_offset, buffer_months):
    """
    Convert a test-year index to month-index bounds [start, end) in ssp_plus_invert.time
    for that single 12-month test year. year_offset=0 is the first year of the test
    period (starts right where the first buffer ends, start_idx + buffer_months);
    year_offset=1 is the second test year (starts exactly 12 months later); and so on.

    Every test year is scored on its own, one year at a time -- MOVING BASELINE: this is
    what lets each test year's reference period (see ref_year_bounds() below) move
    forward in time right along with it.
    """
    s = start_idx + buffer_months + year_offset * 12
    return s, s + 12


def ref_year_bounds(lag, year_offset, buffer_months):
    """
    Convert a reference-year "lag" number to month-index bounds [start, end) in
    ssp_plus_invert.time, for the moving reference period belonging to test year
    `year_offset` (0 = the first test year, 1 = the second, ...).

    There is no zero-weighted gap between the reference period and the test
    year. lag=1 is the 12-month block immediately preceding that test year's own start; lag=2
    is the 12 months before that; and so on.

    The buffer (buffer_months) is not a separate unweighted period -- it's 
    absorbed into whichever reference block(s) it falls into. 

    Moving baseline: for later test years (year_offset > 0), this whole scaffold slides
    forward together with the test year it belongs to -- e.g. for year_offset=1 (the
    second test year), lag=1 is now the entire first test year, lag=2 is whatever was 
    lag=1 for the first test year, and so on.

    Each block is  a full 12 consecutive months, so it always averages over a complete seasonal cycle 
    """
    test_s, _ = test_year_bounds(year_offset, buffer_months)
    e = test_s - (lag - 1) * 12
    return e - 12, e


def popweighted_mean(field, country):
    """
    Population-weighted mean of a gridded field for one country. `field` must have 'lat'/'lon' dims, plus 
    optionally any other dims (e.g. 'ensemble' or 'injection_rate') that should be carried through untouched.

    country_weights_dict['W_norm_eu'] is a sparse (country x gridpoint) matrix where  each row sums to 1 
    and holds that country's population-weighted share of each grid cell. Flatten  `field`'s lat/lon dims 
    into a single gridpoint axis (matching the matrix's column order -- see aggregate_by_country() in 
    Perceptions_functions.py), select this country's single row of the matrix, and matrix-multiply: 
    (1 x n_points) @ (n_points x n_other) -> (1 x n_other), i.e. one pop-weighted number per remaining 
    combination of the other dims.
    """
    row_idx = np.nonzero(countries_idx_eu == country)[0][0]
    weights_row = W_norm_eu.getrow(row_idx)   # sparse (1, n_points)

    other_dims = [dim for dim in field.dims if dim not in ('lat', 'lon')]
    field_t = field.transpose('lat', 'lon', *other_dims)   # lat, lon must be first + adjacent to flatten
    n_points = field_t.sizes['lat'] * field_t.sizes['lon']
    flat = field_t.values.reshape(n_points, -1)              # (n_points, prod(other dims))

    result = np.asarray(weights_row @ flat).ravel()           # (prod(other dims),)
    other_shape = [field_t.sizes[dim] for dim in other_dims]
    result = result.reshape(other_shape)

    return xr.DataArray(result, dims=other_dims,
                        coords={dim: field_t.coords[dim] for dim in other_dims if dim in field_t.coords})


def bg_popweighted_for_window(s, e, country):
    """
    This country's population-weighted mean TREFHT, per ensemble member, over the raw month-index window 
    [s, e) in ssp_plus_invert.time (without CIDER output added). Captures ensemble-to-ensemble 
    internal-variability differences between different years.
    Returns a DataArray with dims (ensemble,). bg='background'
    """
    field = ssp_plus_invert.TREFHT.isel(time=slice(s, e)).mean('time')   # (ensemble, lat, lon)
    return popweighted_mean(field, country)   # (ensemble,)


def cider_delta_popweighted_for_window(s, e, country, injection_lat):
    """
    This country's population-weighted CIDER delta-T over the raw month-index window [s, e) in 
    ssp_plus_invert.time, computed for all injection rates at once 

    CIDER only simulated the first CIDER_HORIZON_MONTHS (60 = 5 years) after deployment begins. 
    Any part of the window that falls before deployment starts, or beyond that 60-month horizon, contributes 
    zero SAI effect. This returns the correctly-scaled mean over just the *covered* months (still averaged 
    over the full window length -- i.e. uncovered months count as 0 in that average), so a window that
    straddles the simulated horizon is handled smoothly.

    This function is general in [s, e) -- it's used for both test-year windows and (moving)
    reference-year windows.
    """
    rel_start, rel_end = s - start_idx, e - start_idx   # months since deployment start
    lo, hi = max(rel_start, 0), min(rel_end, CIDER_HORIZON_MONTHS)
    if hi <= lo:
        return xr.DataArray(np.zeros(cider_combined.sizes['injection_rate']),
                             dims='injection_rate',
                             coords={'injection_rate': cider_combined.injection_rate.values})
    var_cider = cider_combined.sel(injection_lat=injection_lat)   # (injection_rate, time, lat, lon)
    delta_field = var_cider.isel(time=slice(lo, hi)).mean('time')   # (injection_rate, lat, lon) -- mean over covered months only
    frac_covered = (hi - lo) / (rel_end - rel_start)                 # scale down for the uncovered part of the window
    return popweighted_mean(delta_field, country) * frac_covered     # (injection_rate,)


def find_min_injection_rate(country, injection_type, n_test_years, n_ref_years, buffer_months, 
                            ref_weights, min_frac_cooler):
    """
    Find the minimum injection rate (Tg/yr) such that, in at least `min_frac_cooler` of ensemble members, 
    `country`'s population-weighted mean temperature is cooler than its own recency-weighted reference 
    period in every one of the `n_test_years` test years individually.

    Moving baseline, no zero-weighted gap: every individual test year is scored against its own reference 
    period, and that reference period moves forward in time together with the test year it belongs to. 
    
    An injection rate "passes" if a given ensemble member ends up cooler than its own reference period in
    EVERY test year (all `n_test_years` of them individually)

    `buffer_months` sets how long after SAI deployment begins before the first test year starts -- i.e.
    how long people's temperature perceptions accumulate before they're first evaluated. 

    `n_ref_years` controls how many 12-month blocks make up each test year's reference period, counting
    backward from that test year's start 

    `ref_weights` optionally weights the `n_ref_years` reference blocks unequally e.g. to weight recent 
    prior time more heavily than time further in the past. The weights are normalized to sum to 1 internally. 
    `None` weights every reference block equally. 

    Population-weighting is a linear weighted sum, so 
    popweighted_mean(background + delta) == popweighted_mean(background) + popweighted_mean(delta). 
    So that "is this test year cooler than its reference period?" can be split into into two independent 
    pieces that are computed separately and then added, for every test year:
      - `base_diff`: the ensemble-varying, rate-independent natural-variability
         difference between that test year and its reference period (no CIDER effect)
      - `delta_shift`: the rate-varying, ensemble-independent SAI cooling from CIDER over
         that test year vs. its reference period 
    These two pieces are combined into a per-year diff, and a given ensemble member only counts as passing
    if that diff is positive in every individual test year 

    For every test year, a given ensemble member needs to end up cooler than that year's own reference period.

    Returns:
        min_rate (float, or nan if no available rate reaches min_frac_cooler)
        frac_cooler at that rate (float, or nan)
        the full frac_cooler-vs-rate curve, for inspection/plotting if useful later
    """
    if injection_type == 'single_lat':
        injection_lat = lat_to_str_dict[cider_inj_lats_popweighted.sel(
                            country=country).nearest_cider_lat.values.tolist()]
    else: injection_lat = 'uniform'

    ref_lags = [n_ref_years - i for i in range(n_ref_years)]   # oldest (lag=n_ref_years) -> newest (lag=1)

    ## Normalize (or default to uniform) weights over the reference blocks
    if ref_weights is None:
        ref_w = np.ones(n_ref_years) / n_ref_years
    else:
        ref_w = np.asarray(ref_weights, dtype=float)
        assert len(ref_w) == n_ref_years, \
            f"ref_weights has length {len(ref_w)}, expected n_ref_years={n_ref_years}"
        ref_w = ref_w / ref_w.sum()

    ## Score each test year against its own moving reference period, keeping every
    ## year's (base_diff, delta_shift) separate 
    base_diffs, delta_shifts = [], []

    for year_offset in range(n_test_years):
        test_s, test_e = test_year_bounds(year_offset, buffer_months)

        ## base_diff: rate-independent, ensemble-varying 
        bg_test = bg_popweighted_for_window(test_s, test_e, country)                                   # (ensemble,)
        bg_ref = sum(w * bg_popweighted_for_window(*ref_year_bounds(lag, year_offset, buffer_months), country)
                    for w, lag in zip(ref_w, ref_lags))                                                 # (ensemble,)
        base_diffs.append(bg_ref - bg_test)   # (ensemble,) -- positive means this test year is cooler

        ## delta_shift: rate-varying, ensemble-independent 
        delta_test = cider_delta_popweighted_for_window(test_s, test_e, country, injection_lat)         # (injection_rate,)
        delta_ref = sum(w * cider_delta_popweighted_for_window(*ref_year_bounds(lag, year_offset, buffer_months), country, injection_lat)
                        for w, lag in zip(ref_w, ref_lags))                                             # (injection_rate,)
        delta_shifts.append(delta_ref - delta_test)   # (injection_rate,) -- positive means SAI adds extra cooling to this test year

    ## For every (ensemble, injection_rate) pair, check each test year's diff separately and require
    ## all to be positive
    for base_diff, delta_shift in zip(base_diffs, delta_shifts):
        diff = base_diff.values[:, None] + delta_shift.values[None, :]   # (ensemble, injection_rate)
        year_passes.append(diff > 0)

    all_years_pass = np.logical_and.reduce(year_passes)   # (ensemble, injection_rate) -- True only if every test year passed
    frac_cooler_curve = all_years_pass.mean(axis=0)         # (injection_rate,) -- fraction of ensemble members passing every year

    ## Save the lowest injection rate that passes
    rates = cider_combined.injection_rate.values
    meets = frac_cooler_curve >= min_frac_cooler
    if meets.any():
        idx = np.argmax(meets)
        return float(rates[idx]), float(frac_cooler_curve[idx]), frac_cooler_curve
    else:
        return np.nan, np.nan, frac_cooler_curve

Deployment start index: 240 -> 2035-01-16 00:00:00


In [7]:
## 90% 'success' rate swept across three buffer lengths
min_frac_cooler = 0.9

deployment_start_year = 2035
buffer_months_list = [6, 12, 18]   # gap between SAI deployment start and the first test
                                      # year 
# n_test_years = 3               # length (years) of the test period

# ## Moving baseline, no zero-weighted gap: each of the n_test_years test years is scored
# ## against its own reference period

# n_ref_years = 7
# lag_weights = {1: 0.23, 2: 0.20, 3: 0.17, 4: 0.14, 5: 0.11, 6: 0.09, 7: 0.06}
# ref_weights = [lag_weights[lag] for lag in range(n_ref_years, 0, -1)]

# countries_list = list(inj_rates_uniform.country.values)

# injection_types = ['single_lat', 'uniform']

# results_ds_by_buffer = {}   # one xr.Dataset per buffer length, keyed by buffer_months (int)

# for buffer_months in buffer_months_list:
#     print(f'=== buffer_months = {buffer_months} ===')
#     results = []   # one row per (country, injection_type) combination, for this buffer

#     for country in countries_list:
#         print(' ', country)
#         for injection_type in injection_types:
#             min_rate, frac_at_min, curve = find_min_injection_rate(
#                 country, injection_type,
#                 n_test_years=n_test_years,
#                 n_ref_years=n_ref_years,
#                 buffer_months=buffer_months,
#                 ref_weights=ref_weights,
#                 min_frac_cooler=min_frac_cooler)
#             results.append(dict(
#                 country=country, injection_type=injection_type,
#                 min_injection_rate=min_rate, frac_cooler_at_min_rate=frac_at_min))

#     results_df = pd.DataFrame(results)
#     results_ds_by_buffer[buffer_months] = results_df.set_index(
#         ['country', 'injection_type']).to_xarray()

# results_ds_buffer6, results_ds_buffer12, results_ds_buffer18 = (
#     results_ds_by_buffer[6] , results_ds_by_buffer[12], results_ds_by_buffer[18])


In [8]:
# for buffer_months, ds in results_ds_by_buffer.items():
#     ds.to_netcdf(spath + \
#   f'buffer={buffer_months}m_3_test_years_{min_frac_cooler}_cooler_MOVING_BASELINE_YEAR_BY_YEAR_all_countries.nc')

### Reference-period recency weights

The lag-weight table below is implemented directly as `lag_weights` / `ref_weights` above.

Lag (12-month blocks before that test year starts):	Weight (estimated from Moore et al. 2019 Fig. S7)


1:	0.23


2:	0.20


3:	0.17


4:	0.14


5:	0.11


6:	0.09


7:	0.06



## Fixed 12-month buffer, sweeping test-period length (2 / 3 / 4 years)

Lengthening the test period (via `n_test_years`) adds more
test years, each with its own moving reference period, that must all individually pass
for an ensemble member to count as 'successful'. Reference-period length/weights, threshold,
and success fraction are unchanged from the buffer sweep above.


In [10]:
## Fixed 12-month buffer, sweeping the test period length (n_test_years)

buffer_months = 12             # fixed default val -- gap before the first test year begins,

n_test_years_list = [2, 3, 4]  # how many consecutive years, starting immediately after the
                                # first buffer, make up the test period

n_ref_years = 7
lag_weights = {1: 0.23, 2: 0.20, 3: 0.17, 4: 0.14, 5: 0.11, 6: 0.09, 7: 0.06}
ref_weights = [lag_weights[lag] for lag in range(n_ref_years, 0, -1)]

countries_list = list(inj_rates_uniform.country.values)

injection_types = ['single_lat', 'uniform']

results_ds_by_test_years = {}   # one xr.Dataset per test-period length, keyed by n_test_years (int)

for n_test_years in n_test_years_list:
    print(f'=== n_test_years = {n_test_years} ===')
    results = []   # one row per (country, injection_type) combination, for this test-period length

    for country in countries_list:
        print(' ', country)
        for injection_type in injection_types:
            min_rate, frac_at_min, curve = find_min_injection_rate(
                country, injection_type,
                n_test_years=n_test_years,
                n_ref_years=n_ref_years,
                buffer_months=buffer_months,
                ref_weights=ref_weights,
                min_frac_cooler=min_frac_cooler)
            results.append(dict(
                country=country, injection_type=injection_type,
                min_injection_rate=min_rate, frac_cooler_at_min_rate=frac_at_min))

    results_df = pd.DataFrame(results)
    results_ds_by_test_years[n_test_years] = results_df.set_index(
        ['country', 'injection_type']).to_xarray()

=== n_test_years = 3 ===
  Australia
  Bangladesh
  Brazil
  China
  Egypt
  Ethiopia
  European Union
  India
  Indonesia
  Japan
  Nigeria
  Pakistan
  Russia
  Saudi Arabia
  South Africa
  United Kingdom
  United States of America
=== n_test_years = 4 ===
  Australia
  Bangladesh
  Brazil
  China
  Egypt
  Ethiopia
  European Union
  India
  Indonesia
  Japan
  Nigeria
  Pakistan
  Russia
  Saudi Arabia
  South Africa
  United Kingdom
  United States of America


In [11]:
results_ds_test2yr, results_ds_test3yr, results_ds_test4yr = (
    results_ds_by_test_years[2], results_ds_by_test_years[3], results_ds_by_test_years[4])

In [12]:
## Save each test-period length's results to its own file
for n_test_years, ds in results_ds_by_test_years.items():
#     ds.to_netcdf(spath +
#         f'memory_model_buffer={buffer_months}m_{n_ref_years}_ref_years_recency_weighted_'
#         f'ntest={n_test_years}yr.nc')

    ds.to_netcdf(spath + f'12m_buffer_{n_test_years}_test_years_{min_frac_cooler}_cooler_MOVING_BASELINE_YEAR_BY_YEAR_all_countries.nc')

results_ds_by_test_years

In [ ]:
ds.country

In [13]:
results_ds_by_test_years.items()

dict_items([(3, <xarray.Dataset> Size: 696B
Dimensions:                  (country: 17, injection_type: 2)
Coordinates:
  * country                  (country) object 136B 'Australia' ... 'United St...
  * injection_type           (injection_type) object 16B 'single_lat' 'uniform'
Data variables:
    min_injection_rate       (country, injection_type) float64 272B 17.0 ... ...
    frac_cooler_at_min_rate  (country, injection_type) float64 272B 0.902 ......), (4, <xarray.Dataset> Size: 696B
Dimensions:                  (country: 17, injection_type: 2)
Coordinates:
  * country                  (country) object 136B 'Australia' ... 'United St...
  * injection_type           (injection_type) object 16B 'single_lat' 'uniform'
Data variables:
    min_injection_rate       (country, injection_type) float64 272B 19.0 ... ...
    frac_cooler_at_min_rate  (country, injection_type) float64 272B 0.904 ......)])